# Day 2-02｜偵測模型輸出格式預覽
> Python 籃球運動資料分析課程  
> 使用教師提供的 YOLO 權重或範例 JSON，理解 detection 輸出的資料欄位。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 認識 class、confidence、bbox 等 detection 欄位。
- 在沒有模型權重時使用範例資料維持課堂流程。
- 依 confidence 篩選結果並畫回影像。

## 完成產出
- 一張標示高信心偵測框的球場影像。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 確認是否有模型權重。
2. 讀取模型結果或範例 detection JSON。
3. 篩選並視覺化偵測結果。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
# 第一次需要時再打開安裝；課堂如果 Colab 環境已裝好，可以不用跑。
# !pip install -q ultralytics supervision

In [ ]:
from pathlib import Path
from src.cv_utils import read_image_rgb, draw_boxes, show_image, load_json, save_json

IMAGE_PATH = COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png"
MODEL_PATH = COURSE_ROOT / "models" / "detectors" / "yolo26n_basketball_player_best.pt"
image = read_image_rgb(IMAGE_PATH)

detections = []

if MODEL_PATH.exists():
    from ultralytics import YOLO

    model = YOLO(str(MODEL_PATH))
    results = model.predict(str(IMAGE_PATH), conf=0.25, verbose=False)[0]
    names = results.names
    result_boxes = results.boxes
    if result_boxes is not None:
        for box in result_boxes:
            xyxy = box.xyxy[0].cpu().numpy().tolist()
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            detections.append(
                {"bbox_xyxy": xyxy, "class_name": names[cls_id], "confidence": conf}
            )
    print("使用老師提供的 YOLO 權重。")
else:
    data = load_json(
        COURSE_ROOT / "assets" / "samples" / "sample_detections_frame0.json"
    )
    detections = data["detections"]
    print("找不到模型權重，使用 sample detections JSON。")

print("detections:", len(detections))

In [ ]:
# 只畫 confidence 較高的前幾個，避免畫面太亂。
keep = [d for d in detections if d.get("confidence", 0) >= 0.5][:12]
boxes = [d["bbox_xyxy"] for d in keep]
labels = [f"{d['class_name']} {d['confidence']:.2f}" for d in keep]
vis = draw_boxes(image, boxes, labels)
show_image(vis, "detection preview")

out_json = COURSE_ROOT / "assets" / "results" / "d2_02_detection_preview.json"
save_json({"detections": keep}, out_json)
print("saved:", out_json)

觀察問題：哪些人被漏掉？球是否太小？number box 是否容易被誤判？